In [1]:
!pip install numpy pandas matplotlib scikit-learn seaborn


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report, confusion_matrix


In [3]:
!pip install kaggle


In [4]:
from google.colab import files
files.upload()  # Select the kaggle.json file


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"ayushkumarr","key":"b8a6af184d1bd22a87f3c5ddecd47ac7"}'}

In [5]:
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [6]:
!kaggle datasets download -d ankushpanday1/lung-cancer-risk-and-prediction-dataset


Dataset URL: https://www.kaggle.com/datasets/ankushpanday1/lung-cancer-risk-and-prediction-dataset
License(s): Community Data License Agreement - Permissive - Version 1.0
  0% 0.00/15.4M [00:00<?, ?B/s]
100% 15.4M/15.4M [00:00<00:00, 1.29GB/s]


In [7]:
!unzip lung-cancer-risk-and-prediction-dataset.zip

Archive:  lung-cancer-risk-and-prediction-dataset.zip
  inflating: lung_cancer_prediction.csv  


In [9]:
import pandas as pd

# Load the CSV file (replace with actual CSV filename)
df = pd.read_csv('lung_cancer_prediction.csv')

# Check first few rows and info
print(df.head())
print(df.info())
print(df.isnull().sum())


    Country  Age  Gender Smoking_Status Second_Hand_Smoke  \
0    Russia   82    Male  Former Smoker               Yes   
1  Thailand   66  Female  Former Smoker                No   
2  Colombia   87    Male  Former Smoker                No   
3     Egypt   51  Female  Former Smoker                No   
4  DR Congo   43    Male  Former Smoker                No   

  Air_Pollution_Exposure Occupation_Exposure Rural_or_Urban  \
0                 Medium                  No          Urban   
1                   High                  No          Rural   
2                 Medium                  No          Urban   
3                    Low                 Yes          Rural   
4                   High                  No          Urban   

  Socioeconomic_Status Healthcare_Access  ... Treatment_Access  \
0                 High           Limited  ...          Partial   
1               Middle              Good  ...          Partial   
2                  Low              Poor  ...          P

In [10]:
# Display first few rows
print(df.head())

# Show dataset info (columns, data types, null values)
print(df.info())

# Check for missing values
print(df.isnull().sum())

# Summary statistics
print(df.describe())


    Country  Age  Gender Smoking_Status Second_Hand_Smoke  \
0    Russia   82    Male  Former Smoker               Yes   
1  Thailand   66  Female  Former Smoker                No   
2  Colombia   87    Male  Former Smoker                No   
3     Egypt   51  Female  Former Smoker                No   
4  DR Congo   43    Male  Former Smoker                No   

  Air_Pollution_Exposure Occupation_Exposure Rural_or_Urban  \
0                 Medium                  No          Urban   
1                   High                  No          Rural   
2                 Medium                  No          Urban   
3                    Low                 Yes          Rural   
4                   High                  No          Urban   

  Socioeconomic_Status Healthcare_Access  ... Treatment_Access  \
0                 High           Limited  ...          Partial   
1               Middle              Good  ...          Partial   
2                  Low              Poor  ...          P

In [ ]:
# Check column names
print(df.columns)


Index(['Name', 'Surname', 'Age', 'Smokes', 'AreaQ', 'Alkhol', 'Result'], dtype='object')


In [11]:
import numpy as np

df['Mutation_Type'].fillna('Unknown', inplace=True)
df['Treatment_Access'].fillna('Unknown', inplace=True)

# Step 3b: Remove data leakage features (outcomes known AFTER diagnosis)
leakage_cols = ['Mortality_Risk', '5_Year_Survival_Probability', 'Stage_at_Diagnosis', 'Cancer_Type', 'Treatment_Access']
df = df.drop(columns=leakage_cols)

# Step 3c: Encode categorical variables

# Binary features mapping
binary_map = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0, 'Urban': 1, 'Rural': 0}
binary_cols = ['Gender', 'Second_Hand_Smoke', 'Occupation_Exposure', 'Clinical_Trial_Access', 'Language_Barrier',
               'Delay_in_Diagnosis', 'Family_History', 'Indoor_Smoke_Exposure', 'Tobacco_Marketing_Exposure', 'Rural_or_Urban']

for col in binary_cols:
    df[col] = df[col].map(binary_map)

# Ordinal encoding
ordinal_mappings = {
    'Air_Pollution_Exposure': {'Low': 0, 'Medium': 1, 'High': 2},
    'Socioeconomic_Status': {'Low': 0, 'Middle': 1, 'High': 2},
    'Healthcare_Access': {'Poor': 0, 'Limited': 1, 'Good': 2},
    'Insurance_Coverage': {'None': 0, 'Basic': 1, 'Comprehensive': 2},
    'Screening_Availability': {'Low': 0, 'Medium': 1, 'High': 2}
}

for col, mapping in ordinal_mappings.items():
    df[col] = df[col].map(mapping)

# One-hot encoding for nominal features
df = pd.get_dummies(df, columns=['Country', 'Smoking_Status', 'Mutation_Type'], drop_first=True)

# Step 3d: Feature scaling for numerical Age
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df['Age_scaled'] = scaler.fit_transform(df[['Age']])
df = df.drop('Age', axis=1)

# Map target variable if necessary (check if already numeric)
if df['Final_Prediction'].dtype == 'object':
    df['Final_Prediction'] = df['Final_Prediction'].map({'Yes': 1, 'No': 0})

# Final check for NaN in target
print("NaN in target:", df['Final_Prediction'].isna().sum())


/tmp/ipython-input-2334814748.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Mutation_Type'].fillna('Unknown', inplace=True)
/tmp/ipython-input-2334814748.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try

NaN in target: 0


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

X = df.drop('Final_Prediction', axis=1)
y = df['Final_Prediction']

X = X.dropna(axis=1, how='all')

nan_rows = y.isnull()
if nan_rows.any():
    X = X[~nan_rows]
    y = y[~nan_rows]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Missing values in X_train before imputation:")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

# Mean imputation on numeric features
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Convert back to DataFrame
X_train_imputed = pd.DataFrame(X_train_imputed, columns=X_train.columns)
X_test_imputed = pd.DataFrame(X_test_imputed, columns=X_test.columns)

# Apply SMOTE to balance classes in training set
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_imputed, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_balanced.value_counts())


Missing values in X_train before imputation:
Series([], dtype: int64)
Before SMOTE: Final_Prediction
0    294485
1     73748
Name: count, dtype: int64
After SMOTE: Final_Prediction
0    294485
1    294485
Name: count, dtype: int64


In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from imblearn.pipeline import Pipeline

estimators = [
    ('rf', RandomForestClassifier(random_state=42)),
    ('gb', GradientBoostingClassifier(random_state=42)),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
]

pipeline = Pipeline([
    ('voting', VotingClassifier(
        estimators=estimators,
        voting='soft',
        weights=[2, 1, 1],
        n_jobs=-1
    ))
])

param_grid = {
    'voting__rf__n_estimators': [100, 200],
    'voting__rf__max_depth': [None, 10],
    'voting__gb__learning_rate': [0.05, 0.1],
    'voting__gb__n_estimators': [100, 200],
    'voting__lr__C': [0.1, 1.0, 10.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

# Fit using balanced training data with SMOTE already applied
grid_search.fit(X_train_balanced, y_train_balanced)

print("Best parameters:", grid_search.best_params_)
print("Best CV AUC-ROC:", grid_search.best_score_)

# Evaluate on test set
from sklearn.metrics import classification_report, roc_auc_score

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_imputed)
y_proba = best_model.predict_proba(X_test_imputed)[:, 1]

print(classification_report(y_test, y_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))


Fitting 5 folds for each of 48 candidates, totalling 240 fits
